# 92 Veeva Link action hints (Phase 3: B2, B2.1)

Implements `90_` task B2 (rules in `90_` section 6, P3-D5: deterministic rules, one primary hint per HCP by priority, others as secondary) and B2.1 (sense check vs KER-10 outcome at C2). Offline only (P3-D1). Payment columns are never used (P3-D6, M6).

| Priority | Hint | Rule |
| --- | --- | --- |
| 1 | `P2P_SPEAKER` peer-to-peer / speaker programme invitation | `veeva_event_podium_flag_3y` = 1 and `veeva_event_n_inscope_3y` > 0 |
| 2 | `CLINICAL_EVIDENCE` trial investigator | `veeva_trial_n_inscope_5y` > 0 or `veeva_trial_finerenone_flag_5y` = 1 or `veeva_trial_mra_flag_5y` = 1 |
| 3 | `CARDIORENAL_CLASS` class messaging | `veeva_cl_decile_mra_class` >= 8 or `veeva_cl_decile_sglt2_class` >= 8 |
| 4 | `DISEASE_<CKD/HF/T2D>` disease messaging | CL disease decile >= 8 |
| 5 | `SCIENTIFIC_CONTENT` digital content | `veeva_dr_any_inscope` = 1 or `veeva_pub_n_inscope_5y` > 0 |

Inputs: B0 list base (`91_`), T1 at the cohort (`71_`). Outputs: `OUT/business/b2_action_hints/run=<ts>/{list, coverage, overlap, sense_check}`; checks M5 to M7.

In [0]:
from pyspark.sql import functions as F
import pandas as pd, numpy as np, datetime as dt, json

C = "`ph-com-ai-us-prod-catalog-2553213839599380`"
OUT = "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_1_volume/hcp_level_model/dev_pipeline/notebook_output/veeva_link_to_people/"
P1, P2, PB = OUT + "features_phase1/", OUT + "features_phase2/", OUT + "business/"
MV = f"{C}.kerendia_metric_views.mv_kerendia_nbrx"
B0_RUN = "20261123T103200"   # 91_ run
RUN_TS = dt.datetime.now(dt.timezone.utc).strftime("%Y%m%dT%H%M%S")

T1C = ["veeva_event_podium_flag_3y", "veeva_event_n_inscope_3y", "veeva_trial_n_inscope_5y", "veeva_trial_pi_flag_5y",
       "veeva_trial_finerenone_flag_5y", "veeva_trial_mra_flag_5y", "veeva_pub_n_inscope_5y"]
T3C = ["veeva_cl_decile_mra_class", "veeva_cl_decile_sglt2_class", "veeva_cl_decile_ckd", "veeva_cl_decile_hf", "veeva_cl_decile_t2d", "veeva_dr_any_inscope"]
HINTS = ["P2P_SPEAKER", "CLINICAL_EVIDENCE", "CARDIORENAL_CLASS", "DISEASE_CKD", "DISEASE_HF", "DISEASE_T2D", "SCIENTIFIC_CONTENT"]   # priority order

def hints(d):
    g = lambda c: d[c].fillna(0)
    h = pd.DataFrame(index=d.index)
    h["P2P_SPEAKER"] = (g("veeva_event_podium_flag_3y") == 1) & (g("veeva_event_n_inscope_3y") > 0)
    h["CLINICAL_EVIDENCE"] = (g("veeva_trial_n_inscope_5y") > 0) | (g("veeva_trial_finerenone_flag_5y") == 1) | (g("veeva_trial_mra_flag_5y") == 1)
    h["CARDIORENAL_CLASS"] = (g("veeva_cl_decile_mra_class") >= 8) | (g("veeva_cl_decile_sglt2_class") >= 8)
    for k in ("CKD", "HF", "T2D"):
        h[f"DISEASE_{k}"] = g(f"veeva_cl_decile_{k.lower()}") >= 8
    h["SCIENTIFIC_CONTENT"] = (g("veeva_dr_any_inscope") == 1) | (g("veeva_pub_n_inscope_5y") > 0)
    h = h.astype(int)
    out = d.copy()
    out["PRIMARY_HINT"] = h[HINTS].idxmax(axis=1).where(h.sum(axis=1) > 0)
    out["SECONDARY_HINTS"] = h.apply(lambda r: ", ".join(k for k in HINTS if r[k]), axis=1)
    out["SECONDARY_HINTS"] = [", ".join(x for x in s.split(", ") if x and x != p) for s, p in zip(out.SECONDARY_HINTS, out.PRIMARY_HINT.fillna(""))]
    out["CLINICAL_EVIDENCE_PI"] = ((h.CLINICAL_EVIDENCE == 1) & (g("veeva_trial_pi_flag_5y") == 1)).astype(int)
    return out, h

def load(cohort, base_cols):
    t1 = spark.read.parquet(P1 + f"t1_t2/COHORT_ID={cohort}").select("NPI_NUMBER", *T1C)
    t3 = spark.read.parquet(P2 + f"t3/version=snapshot/COHORT_ID={cohort}").select("NPI_NUMBER", "PROVIDER_ID", "veeva_coverage_status", "veeva_best_decile", *T3C)
    return t3.join(t1, "NPI_NUMBER", "left").toPandas()

b0 = spark.read.parquet(PB + f"b0_list_base/run={B0_RUN}/list").select("NPI_NUMBER", "HCP_FIRST_NAME", "HCP_LAST_NAME", "SPECIALTY", "CITY", "STATE", "SEGMENT",
                                                                     "KER10_WRITER_12M", "TIER", "INDICATIONS", "veeva_score_from_stale_load").toPandas()
d22 = load("2026-10_offset7", None).merge(b0, on="NPI_NUMBER", how="left")
d22, h22 = hints(d22)
b2 = d22[d22.PRIMARY_HINT.notna()].copy()
print(f"NPIs {len(d22):,} | with any hint {len(b2):,} ({len(b2) / len(d22):.1%}) | with hint, Veeva-scored {int((b2.veeva_coverage_status == 'SCORED').sum()):,}")
coverage = pd.DataFrame({"hint": HINTS, "priority": range(1, len(HINTS) + 1),
                         "hcps_rule_fires": [int(h22[k].sum()) for k in HINTS],
                         "hcps_primary": [int((d22.PRIMARY_HINT == k).sum()) for k in HINTS]})
display(coverage)
print("Primary hint by segment (HCPs):")
seg_cov = pd.crosstab(b2.PRIMARY_HINT, b2.SEGMENT).reindex(HINTS)
display(seg_cov.reset_index())
ov = (h22[HINTS].T @ h22[HINTS])
print("Overlap (HCPs where both rules fire):")
display(ov.reset_index().rename(columns={"index": "hint"}))

In [0]:
# C2 cut-off 2025-03-06, outcome = KER-10 writer in 2025-03-07..2026-03-06, segments as in R4 (83_)
d2 = load("2025-02_offset7", None)
d2, _ = hints(d2)
seg2 = b0[["NPI_NUMBER", "SEGMENT"]]
w_out = spark.sql(f"""SELECT `provider_id` AS PROVIDER_ID, MEASURE(`active_nbrx_writers`) AS W FROM {MV}
                     WHERE `week_ending` >= '2025-03-07' AND `week_ending` <= '2026-03-06' GROUP BY ALL""").filter("W > 0").select("PROVIDER_ID").toPandas()
w_pre = spark.sql(f"""SELECT `provider_id` AS PROVIDER_ID, MEASURE(`paid_nbrx`) AS N FROM {MV}
                     WHERE `week_ending` >= '2024-03-07' AND `week_ending` <= '2025-03-06' GROUP BY ALL""").filter("N > 0").select("PROVIDER_ID").toPandas()
d2 = d2.merge(seg2, on="NPI_NUMBER", how="left")
d2["OUT_WRITER"] = d2.PROVIDER_ID.isin(w_out.PROVIDER_ID).astype(int)
d2["NEW_ELIG"] = (~d2.PROVIDER_ID.isin(w_pre.PROVIDER_ID)).astype(int)
d2["HINT"] = d2.PRIMARY_HINT.fillna("(no hint)")
seg_avg = d2.groupby("SEGMENT").agg(SEG_RATE=("OUT_WRITER", "mean"))
seg_new = d2[d2.NEW_ELIG == 1].groupby("SEGMENT").agg(SEG_NEW_RATE=("OUT_WRITER", "mean"))
sc = d2.groupby(["SEGMENT", "HINT"]).agg(HCPS=("OUT_WRITER", "size"), RATE=("OUT_WRITER", "mean"))
sc = sc.join(d2[d2.NEW_ELIG == 1].groupby(["SEGMENT", "HINT"]).agg(NEW_HCPS=("OUT_WRITER", "size"), NEW_RATE=("OUT_WRITER", "mean")))
sc = sc.reset_index().merge(seg_avg.join(seg_new).reset_index(), on="SEGMENT")
for c in ("RATE", "NEW_RATE", "SEG_RATE", "SEG_NEW_RATE"):
    sc[c] = (100 * sc[c]).round(2)
sc["BELOW_SEGMENT_AVG"] = ((sc.HINT != "(no hint)") & (sc.HCPS >= 100) & (sc.RATE < sc.SEG_RATE)).astype(int)
print(f"C2 NPIs {len(d2):,} | outcome writers {int(d2.OUT_WRITER.sum()):,} | hints identical to C22 for {(d2.set_index('NPI_NUMBER').HINT.reindex(d22.NPI_NUMBER).fillna('(no hint)').values == d22.PRIMARY_HINT.fillna('(no hint)').values).mean():.2%} of NPIs")
print("All segments, by primary hint:")
all_ = d2.groupby("HINT").agg(HCPS=("OUT_WRITER", "size"), RATE=("OUT_WRITER", "mean"))
all_["RATE"] = (100 * all_.RATE).round(2)
display(all_.reindex(HINTS + ["(no hint)"]).reset_index())
print("Cells (segment x hint, >= 100 HCPs) below their segment average:")
display(sc[sc.BELOW_SEGMENT_AVG == 1][["SEGMENT", "HINT", "HCPS", "RATE", "SEG_RATE", "NEW_RATE", "SEG_NEW_RATE"]])

In [0]:
def write(df, path):
    x = df.copy()
    for c in x.columns:
        x[c] = x[c].astype(str).replace({"nan": None, "None": None}) if x[c].dtype == object else x[c].astype(float)
    x["run_ts"] = RUN_TS
    spark.createDataFrame(x).write.mode("overwrite").parquet(path)

LIST_COLS = ["NPI_NUMBER", "PROVIDER_ID", "HCP_FIRST_NAME", "HCP_LAST_NAME", "SPECIALTY", "CITY", "STATE", "SEGMENT", "veeva_coverage_status", "veeva_best_decile",
             "TIER", "INDICATIONS", "KER10_WRITER_12M", "PRIMARY_HINT", "SECONDARY_HINTS", "CLINICAL_EVIDENCE_PI", "veeva_score_from_stale_load", *T1C, *T3C]
P = f"{PB}b2_action_hints/run={RUN_TS}/"
write(b2[LIST_COLS], P + "list"); write(coverage, P + "coverage"); write(ov.reset_index().rename(columns={"index": "hint"}), P + "overlap"); write(sc, P + "sense_check")

def lock(name):
    j = json.loads("".join(r.value for r in spark.read.text(name).collect()))
    j = j.get("columns", j) if isinstance(j, dict) else j
    return {c if isinstance(c, str) else c.get("name") for c in j}
locked = lock(P1 + "schema_lock/columns.json") | lock(P2 + "schema_lock/t3_columns.json")
rule_cols = set(T1C) | set(T3C)
chk = pd.DataFrame([
    {"check": "M5", "name": "every rule column is in a schema lock", "value": sorted(rule_cols - locked), "pass": rule_cols <= locked},
    {"check": "M6", "name": "no payment column used or exported", "value": sorted(c for c in set(LIST_COLS) | rule_cols if "pay" in c.lower()),
     "pass": not any("pay" in c.lower() for c in set(LIST_COLS) | rule_cols)},
    {"check": "M7", "name": "hint rows are in Veeva", "value": b2.veeva_coverage_status.value_counts().to_dict(), "pass": bool((b2.veeva_coverage_status != "NOT_IN_VEEVA").all())},
    {"check": "M7b", "name": "one primary hint per hint row", "value": int(b2.PRIMARY_HINT.isna().sum()), "pass": bool(b2.PRIMARY_HINT.notna().all() and b2.NPI_NUMBER.is_unique)}])
chk["value"] = chk.value.astype(str); chk["run_ts"] = RUN_TS
spark.createDataFrame(chk.astype(str)).write.mode("overwrite").parquet(f"{PB}checks/run={RUN_TS}/b2")
display(chk)
print("ALL PASS:", bool(chk["pass"].all()), "| run:", RUN_TS, "| written:", P)